#### TF-IDF 계산법

In [126]:
docs = [
    '영화가 너무 재미있었다',
    '영화가 너무 지루하다',
    '배우의 연기가 너무 좋았다'
]

In [127]:
tokens = [ doc.split() for doc in docs ]

In [128]:
tokens

[['영화가', '너무', '재미있었다'], ['영화가', '너무', '지루하다'], ['배우의', '연기가', '너무', '좋았다']]

In [129]:
# 단어 사전 생성 → 데이터를 1차원으로 변경하고 중복 데이터를 제거

vocab1 = []

for token in tokens:
    for word in token:
        vocab1.append(word)
    

# 리스트에서 중복 값 제거 → 집합의 형태로 변환했다가 다시 리스트로 변환
vocab1 = list( set( vocab1 ) )
vocab1

['재미있었다', '영화가', '배우의', '너무', '좋았다', '지루하다', '연기가']

In [130]:
# []는 더할 형태를 지정해주는 것
sum(tokens, [])

['영화가', '너무', '재미있었다', '영화가', '너무', '지루하다', '배우의', '연기가', '너무', '좋았다']

In [131]:
vocab = list(set(sum(tokens, [])))

print(vocab)

['재미있었다', '영화가', '배우의', '너무', '좋았다', '지루하다', '연기가']


In [132]:
# TF, IDF 계산

import math

In [133]:
# 단어 사전의 길이
V = len(vocab)

# 전체 문서의 길이
N = len(docs)

In [134]:
# 단어의 개수를 생성

word_cnt = {
    w: sum(1 for doc in tokens if w in doc) for w in vocab
}

word_cnt

{'재미있었다': 1, '영화가': 2, '배우의': 1, '너무': 3, '좋았다': 1, '지루하다': 1, '연기가': 1}

In [135]:
# TF 계산식 함수

def tf(word, doc):
    # word: 단어 사전의 각 원소들 대입
    # doc: tokens의 각 원소들 대입
    result = math.log(doc.count(word) + 1)
    # doc.count(word): 문장에서 특정 단어의 개수
    # len(doc): 문장의 단어의 개수
    return result

In [136]:
# IDF 계산식 함수

def idf(word):
    # word: 단어 사전의 각 원소
    result = math.log( (N) / (word_cnt[word] + 1) ) + 1
    # N: docs의 길이 → 문장들의 개수
    # word_cnt[word]: 전체 문서에서 특정 단어의 개수
    return result

In [137]:
X_tfidf = [
    [ tf(w, doc) * idf(w) for w in vocab ] for doc in tokens
]

In [138]:
X_tfidf

[[0.9741941770605529,
  0.6931471805599453,
  0.0,
  0.49374116314235145,
  0.0,
  0.0,
  0.0],
 [0.0,
  0.6931471805599453,
  0.0,
  0.49374116314235145,
  0.0,
  0.9741941770605529,
  0.0],
 [0.0,
  0.0,
  0.9741941770605529,
  0.49374116314235145,
  0.9741941770605529,
  0.0,
  0.9741941770605529]]

In [139]:
import pandas as pd

In [140]:
pd.DataFrame(X_tfidf, columns = vocab)

,재미있었다,영화가,배우의,너무,좋았다,지루하다,연기가
0,0.974194,0.693147,0.000000,0.493741,0.000000,0.000000,0.000000
1,0.000000,0.693147,0.000000,0.493741,0.000000,0.974194,0.000000
2,0.000000,0.000000,0.974194,0.493741,0.974194,0.000000,0.974194


In [141]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [142]:
vec = TfidfVectorizer(
    ngram_range = (1, 1),
    min_df = 1
)

In [143]:
X = vec.fit_transform(docs)

In [144]:
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())

,너무,배우의,연기가,영화가,재미있었다,좋았다,지루하다
0,0.425441,0.000000,0.000000,0.547832,0.720333,0.000000,0.000000
1,0.425441,0.000000,0.000000,0.547832,0.000000,0.000000,0.720333
2,0.322745,0.546454,0.546454,0.000000,0.000000,0.546454,0.000000


#### LSA(잠재 의미 분석)

- 문서 안에서 단어 사이의 잠재적인 의미 구조를 추출하는 기법
- TF-IDF 방식은 단어 간의 의미적 유사성을 반영하지 않은 것
- TF-IDF 방식에서 SVD 분해하여 단어 간의 의미를 파악
- 차원 축소를 통해서 관계성을 확인

<br>

- **LSA 효과**
    - 벡터 공간의 차원을 줄여서 계산 효율 증가
    - '영화', '필름' 등 비슷한 문맥의 단어를 가까운 벡터로 이동 (의미 유추)
    - 문서들을 주제별로 분류 가능 (토픽 분석)

<br>

- **TruncatedSVD** (차원 축소 모델)
    - 절단된 특이값을 분해
    - 고차원 희소행렬(값이 0인 행렬)을 낮은 차원으로 압축하여 데이터의 구조적 의미를 유지
    - 자연어 처리, 추천 시스템, 의미 분석, 잠재적인 토픽 분석에서 주로 사용
    - TF-IDF 행렬은 고차원 → 저차원
    - 0으로 이루어진 희소행렬들을 구조적인 의미를 유지하면서 값들을 부여
    - 같은 토픽의 문서는 같은 벡터 공간에서 가깝게 위치 → 유사도 기반 자연어 처리에서 활용

In [145]:
from sklearn.decomposition import TruncatedSVD
from konlpy.tag import Okt

In [146]:
okt = Okt()

def tokenize(text):
    result = []
    for word, pos in okt.pos(text):
        if pos in ['Noun', 'Adjective', 'Verb']:
            result.append(word)
    return result

In [147]:
docs = [
    '이 영화가 정말 재미있었다',
    '매우 연기가 뛰어나다',
    '이 영화 별로다',
    '지루한 영화는 보기 어렵다',
    '정말 훌륭한 연기였다',
    '연기가 별로라서 지루했다'
]

In [148]:
tfidf = TfidfVectorizer(
    tokenizer = tokenize,
    ngram_range = (1, 1),
    min_df = 1,
    max_df = 0.8,
    lowercase = False
)

In [149]:
X_tfidf = tfidf.fit_transform(docs)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [150]:
X_tfidf.shape

(6, 14)

In [151]:
# 차원 축소
lsa = TruncatedSVD(n_components = 2, random_state = 42)
X_lsa = lsa.fit_transform(X_tfidf)

In [152]:
df_lsa = pd.DataFrame(X_lsa, columns = ['topic1', 'topic2'])
df_lsa

,topic1,topic2
0,0.722615,-0.336336
1,0.229990,0.678032
2,0.801574,-0.279071
3,0.346712,-0.383244
4,0.392081,0.481308
5,0.516723,0.493418


In [153]:
df_lsa['document'] = docs
df_lsa

,topic1,topic2,document
0,0.722615,-0.336336,이 영화가 정말 재미있었다
1,0.229990,0.678032,매우 연기가 뛰어나다
2,0.801574,-0.279071,이 영화 별로다
3,0.346712,-0.383244,지루한 영화는 보기 어렵다
4,0.392081,0.481308,정말 훌륭한 연기였다
5,0.516723,0.493418,연기가 별로라서 지루했다


In [154]:
features = tfidf.get_feature_names_out()
features

array(['뛰어나다', '매우', '별로', '보기', '어렵다', '연기', '였다', '영화', '이', '재미있었다',
       '정말', '지루한', '지루했다', '훌륭한'], dtype=object)

In [155]:
components = lsa.components_
components

array([[ 0.0830607 ,  0.0830607 ,  0.44101117,  0.10569975,  0.10569975,
         0.28313222,  0.12558933,  0.47611211,  0.47725902,  0.24451986,
         0.30349483,  0.10569975,  0.20031593,  0.12558933],
       [ 0.33833856,  0.33833856,  0.0835962 , -0.161434  , -0.161434  ,
         0.56468446,  0.21301717, -0.33302649, -0.26207738, -0.15725175,
         0.04572844, -0.161434  ,  0.26429403,  0.21301717]])

In [156]:
components.shape

(2, 14)

In [157]:
pd.DataFrame(components, index = ['topic1', 'topic2'], columns = features).T

,topic1,topic2
뛰어나다,0.083061,0.338339
매우,0.083061,0.338339
별로,0.441011,0.083596
보기,0.105700,-0.161434
어렵다,0.105700,-0.161434
연기,0.283132,0.564684
였다,0.125589,0.213017
영화,0.476112,-0.333026
이,0.477259,-0.262077
재미있었다,0.244520,-0.157252


1. rating_train.txt 파일 로드
2. 결측치 제외, id 컬럼 제외
3. documnet의 중복 데이터 제거
4. 상위 5,000개 데이터 추출
5. 독립종속 분할, train test 분할
6. X_train 이용 tfidf, lsa 작업
7. SVC 모델 이용하여 성능 평가

In [158]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [159]:
df = pd.read_csv('../data/ratings_train.txt', sep = '\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [160]:
df.dropna(inplace = True)
df.drop('id', axis = 1, inplace = True)
df.info()

<class 'pandas.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   document  149995 non-null  str  
 1   label     149995 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.4 MB


In [161]:
df.drop_duplicates('document', inplace = True)

In [162]:
df2 = df.iloc[:5000]

In [163]:
df2['label'].value_counts()

label
0    2502
1    2498
Name: count, dtype: int64

In [164]:
X = df2['document'].values
y = df2['label'].values

In [165]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

In [166]:
okt = Okt()

def tokenize(text):
    return okt.morphs(text)

vec = TfidfVectorizer(
    tokenizer = tokenize,
    lowercase = False,
    ngram_range = (1, 2),
    min_df = 3
)

In [167]:
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [168]:
svc = SVC(
    kernel = 'linear',
    C = 1.0,
    random_state = 42
)

In [169]:
X_train_vec.shape

(4000, 4730)

In [170]:
svc.fit(X_train_vec, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [171]:
pred_tfidf = svc.predict(X_test_vec)

In [172]:
print(classification_report(pred_tfidf, y_test))

              precision    recall  f1-score   support

           0       0.79      0.75      0.77       528
           1       0.73      0.78      0.75       472

    accuracy                           0.76      1000
   macro avg       0.76      0.76      0.76      1000
weighted avg       0.76      0.76      0.76      1000



In [173]:
# SVD를 이용해서 차원 축소 → 200개의 feature로 축소
lsa = TruncatedSVD(
    n_components = 200,
    random_state = 42
)

In [174]:
X_train_lsa = lsa.fit_transform(X_train_vec)
X_test_lsa = lsa.transform(X_test_vec)

In [175]:
X_train_lsa.shape

(4000, 200)

In [176]:
svc.fit(X_train_lsa, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [177]:
pred_lsa = svc.predict(X_test_lsa)

In [178]:
print(classification_report(pred_lsa, y_test))

              precision    recall  f1-score   support

           0       0.75      0.71      0.73       525
           1       0.70      0.73      0.72       475

    accuracy                           0.72      1000
   macro avg       0.72      0.72      0.72      1000
weighted avg       0.72      0.72      0.72      1000



In [179]:
pd.DataFrame(X_train_lsa)

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
0,0.066284,-0.042363,0.012952,0.322909,-0.149863,-0.427388,0.639381,-0.327136,0.096032,-0.250294,...,0.001638,-0.001803,-0.008907,-0.008075,0.005508,-0.006507,-0.001932,-0.002540,-0.006923,-0.002659
1,0.043724,-0.009106,0.006832,-0.040717,0.002555,-0.018963,0.073149,0.101102,-0.002680,0.105653,...,0.014595,0.074100,-0.053171,0.066752,-0.058986,-0.014924,0.079256,0.077041,-0.038936,0.007742
2,0.166865,-0.074597,0.081214,-0.159094,-0.342219,-0.010742,-0.116317,-0.188821,-0.026707,0.087651,...,-0.016888,0.001592,0.008867,0.021722,0.019809,0.019332,0.008699,0.037256,0.018223,-0.002513
3,0.054177,-0.023260,0.034290,-0.015745,-0.110448,0.017660,-0.051218,-0.041615,0.188922,0.029380,...,-0.027908,0.037902,0.020806,-0.018710,-0.008632,0.041436,-0.059002,-0.006368,0.028651,0.009382
4,0.005664,-0.004106,-0.002343,-0.004398,-0.002969,0.002713,0.002858,0.002746,0.003702,0.001785,...,0.004457,0.028511,-0.043387,-0.067354,0.021137,0.017367,0.012637,-0.051232,-0.084897,0.023764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,0.116555,0.089018,-0.020101,-0.068399,-0.018275,-0.026304,-0.021618,-0.078150,-0.063910,-0.005106,...,-0.030442,0.013202,0.029160,0.009389,-0.010076,-0.019705,0.020027,0.001302,-0.020941,-0.030243
3996,0.216761,0.087835,-0.012917,0.008541,0.023487,0.044170,-0.010286,0.000707,0.037685,-0.027659,...,-0.019803,-0.006470,-0.001417,0.023241,0.030959,0.009795,0.006794,-0.043949,-0.043432,0.028453
3997,0.077075,-0.027844,0.032116,-0.054863,-0.143533,-0.006758,-0.057704,-0.056373,0.089408,0.038684,...,0.007076,0.037223,0.012187,0.028205,-0.014683,0.014992,0.023116,-0.033490,0.024224,-0.012595
3998,0.104292,-0.010974,0.017166,0.088516,0.006210,0.175031,0.011483,-0.022483,0.026128,-0.065783,...,-0.020896,0.009027,-0.023018,0.006040,0.000664,-0.027642,0.022494,-0.015012,0.016050,0.034697


In [180]:
from sklearn.preprocessing import StandardScaler

In [181]:
std = StandardScaler()

In [182]:
X_train_sc = std.fit_transform(X_train_lsa)
X_test_sc = std.transform(X_test_lsa)

In [183]:
svc.fit(X_train_sc, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [184]:
pred_sc = svc.predict(X_test_sc)

In [185]:
print(classification_report(pred_sc, y_test))

              precision    recall  f1-score   support

           0       0.77      0.73      0.75       529
           1       0.72      0.76      0.74       471

    accuracy                           0.74      1000
   macro avg       0.74      0.75      0.74      1000
weighted avg       0.75      0.74      0.75      1000



In [186]:
# TF-IDF에서 MaxAbsScaler를 사용하고 모델 성능 평가

from sklearn.preprocessing import MaxAbsScaler

In [187]:
ma = MaxAbsScaler()

In [188]:
X_train_ma = ma.fit_transform(X_train_vec)
X_test_ma = ma.transform(X_test_vec)

In [189]:
svc.fit(X_train_ma, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [190]:
pred_ma = svc.predict(X_test_ma)

In [191]:
print(classification_report(pred_ma, y_test))

              precision    recall  f1-score   support

           0       0.75      0.73      0.74       509
           1       0.73      0.74      0.73       491

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000

